# Drug Type Classification

**Classification Project 19 of 100** - Predict the appropriate drug type from clinical features.

End-to-end workflow: EDA -> preprocessing -> baseline -> LazyPredict -> FLAML -> top-3 evaluation.

## 2. Project Overview

Prescribing the right drug depends on patient characteristics. This notebook builds a 5-class classifier predicting drug type using the Drug Classification dataset (200 rows, 5 features, 5 classes). An easy dataset with strong patterns.

## 3. Learning Objectives

1. Small multiclass dataset (200 rows, 5 classes)
2. Handle ordinal categoricals (BP: LOW/NORMAL/HIGH)
3. Use OrdinalEncoder for ordered features
4. Evaluate with macro-F1 and per-class metrics
5. Understand Na_to_K ratio dominance
6. Benchmark with LazyPredict and FLAML
7. Appreciate overfitting risk on small data

## 4. Problem Statement

> Given age, sex, BP, cholesterol, and Na_to_K ratio, predict which of 5 drugs to prescribe.

5-class classification. Macro-F1 is the primary metric.

## 5. Why This Project Matters

| Stakeholder | Impact |
|---|---|
| Physicians | Decision support |
| Pharmacists | Validate prescriptions |
| ML learners | Multiclass with ordinal features |
| Healthcare | Feature-drug relationships |

## 6. Dataset Overview

| Property | Value |
|---|---|
| Kaggle slug | prathamtripathi/drug-classification |
| Rows | 200 |
| Features | 5 (Age, Sex, BP, Cholesterol, Na_to_K) |
| Target | Drug (drugA-drugY) |
| Classes | 5 (drugY most common ~45%) |

## 7. Dataset Source and License Notes

- Kaggle: https://www.kaggle.com/datasets/prathamtripathi/drug-classification
- License: CC0 Public Domain.
- Small educational dataset.

## 8. Environment Setup

In [ ]:
import subprocess, sys, importlib
for pkg in ["lazypredict","flaml","kagglehub","xgboost","lightgbm"]:
    try: importlib.import_module(pkg)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Environment ready.")

## 9. Imports

In [ ]:
import os, warnings, glob
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, f1_score, ConfusionMatrixDisplay, classification_report)
from lazypredict.Supervised import LazyClassifier
from flaml import AutoML
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid"); SEED=42
print("All imports loaded.")

## 10. Configuration / Constants

In [ ]:
KAGGLE_SLUG = "prathamtripathi/drug-classification"
TARGET = "Drug"
TEST_SIZE = 0.15; VAL_SIZE = 0.15; SEED = 42; FLAML_BUDGET = 60
print(f"Target: {TARGET}")

## 11. Dataset Download or Loading

In [ ]:
import kagglehub
try:
    path = kagglehub.dataset_download(KAGGLE_SLUG)
    print(f"Downloaded to: {path}")
except Exception as e:
    raise RuntimeError(f"Download failed: {e}")
csv_files = glob.glob(os.path.join(path,"**","*.csv"), recursive=True)
df_raw = pd.read_csv(csv_files[0])
print(f"Shape: {df_raw.shape}")
df_raw.head()

## 12. Data Validation Checks

In [ ]:
print(f"Missing: {df_raw.isnull().sum().sum()}")
df = df_raw.copy()
dupes = df.duplicated().sum()
print(f"Duplicates: {dupes}")
if dupes > 0: df = df.drop_duplicates().reset_index(drop=True)
assert TARGET in df.columns
print(f"\nTarget:\n{df[TARGET].value_counts()}")

## 13. Exploratory Data Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(6,4))
df[TARGET].value_counts().plot.bar(ax=ax, color=sns.color_palette("Set2", 5))
ax.set_title("Drug Type Distribution")
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if "Na_to_K" in df.columns:
    for drug in df[TARGET].unique():
        axes[0].hist(df[df[TARGET]==drug]["Na_to_K"], bins=20, alpha=0.5, label=drug)
    axes[0].set_title("Na_to_K by Drug"); axes[0].legend(fontsize=7)
if "Age" in df.columns:
    for drug in df[TARGET].unique():
        axes[1].hist(df[df[TARGET]==drug]["Age"], bins=20, alpha=0.5, label=drug)
    axes[1].set_title("Age by Drug"); axes[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

In [ ]:
cat_show = [c for c in ["BP","Cholesterol","Sex"] if c in df.columns]
if cat_show:
    fig, axes = plt.subplots(1, len(cat_show), figsize=(5*len(cat_show),4))
    if len(cat_show)==1: axes=[axes]
    for ax, col in zip(axes, cat_show):
        ct = pd.crosstab(df[col], df[TARGET], normalize="index")
        ct.plot.bar(stacked=True, ax=ax, colormap="Set2")
        ax.set_title(f"Drug by {col}"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## 14. Target Analysis

5 classes, drugY most common (~45%). Na_to_K ratio almost perfectly separates drugY.

In [ ]:
print(df[TARGET].value_counts(normalize=True))
print(f"\nMajority baseline: {df[TARGET].value_counts(normalize=True).max():.1%}")

## 15. Train / Validation / Test Split

In [ ]:
le = LabelEncoder()
X = df.drop(columns=[TARGET]); y = le.fit_transform(df[TARGET])
X_tmp, X_test, y_tmp, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tmp, y_tmp, test_size=VAL_SIZE/(1-TEST_SIZE), random_state=SEED, stratify=y_tmp)
print(f"Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}")
print(f"Classes: {le.classes_}")

## 16. Preprocessing Strategy

- Numeric (Age, Na_to_K): StandardScaler
- Ordinal (BP, Cholesterol): OrdinalEncoder
- Nominal (Sex): OneHotEncoder

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
ord_cols = [c for c in ["BP"] if c in X_train.columns]
chol_cols = [c for c in ["Cholesterol"] if c in X_train.columns]
nom_cols = [c for c in ["Sex"] if c in X_train.columns]
transformers = [("num", StandardScaler(), num_cols)]
if ord_cols: transformers.append(("ord_bp", OrdinalEncoder(categories=[["LOW","NORMAL","HIGH"]]), ord_cols))
if chol_cols: transformers.append(("ord_ch", OrdinalEncoder(categories=[["NORMAL","HIGH"]]), chol_cols))
if nom_cols: transformers.append(("nom", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nom_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print(f"Num: {num_cols} | Ord: {ord_cols+chol_cols} | Nom: {nom_cols}")

## 17. Feature Engineering

- NA_K_HIGH: binary flag for Na_to_K > 14.6
- AGE_GROUP: binned age

In [ ]:
def eng(d):
    d = d.copy()
    if "Na_to_K" in d.columns: d["NA_K_HIGH"] = (d["Na_to_K"]>14.6).astype(float)
    if "Age" in d.columns: d["AGE_GROUP"] = pd.cut(d["Age"], bins=[0,30,50,100], labels=False).astype(float)
    return d
X_train=eng(X_train); X_val=eng(X_val); X_test=eng(X_test)
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
transformers = [("num", StandardScaler(), num_cols)]
if ord_cols: transformers.append(("ord_bp", OrdinalEncoder(categories=[["LOW","NORMAL","HIGH"]]), ord_cols))
if chol_cols: transformers.append(("ord_ch", OrdinalEncoder(categories=[["NORMAL","HIGH"]]), chol_cols))
if nom_cols: transformers.append(("nom", OneHotEncoder(handle_unknown="ignore",sparse_output=False), nom_cols))
preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")
print(f"Features: {X_train.shape[1]}")

## 18. Baseline Model

In [ ]:
results = {}
for name, clf in [("Dummy",DummyClassifier(strategy="most_frequent",random_state=SEED)),
                  ("LogReg",LogisticRegression(max_iter=1000,random_state=SEED)),
                  ("RF",RandomForestClassifier(n_estimators=200,random_state=SEED,n_jobs=-1))]:
    pipe = Pipeline([("pre",preprocessor),("clf",clf)])
    pipe.fit(X_train, y_train); yp=pipe.predict(X_val)
    r={"Accuracy":accuracy_score(y_val,yp),"Macro-F1":f1_score(y_val,yp,average="macro")}
    results[name]=r; print(f"{name}: {r}")

## 19. LazyPredict Benchmark

In [ ]:
X_train_lp=preprocessor.fit_transform(X_train); X_val_lp=preprocessor.transform(X_val)
lazy=LazyClassifier(verbose=0,ignore_warnings=True,random_state=SEED)
models_lp,_=lazy.fit(X_train_lp,X_val_lp,y_train,y_val)
models_lp.head(15)

## 20. FLAML AutoML Run

In [ ]:
X_fl=X_train.copy(); X_vfl=X_val.copy()
for c in ord_cols+chol_cols+nom_cols:
    if c in X_fl.columns: X_fl[c]=X_fl[c].astype(str); X_vfl[c]=X_vfl[c].astype(str)
automl=AutoML()
automl.fit(X_fl,y_train,task="classification",metric="macro_f1",time_budget=FLAML_BUDGET,seed=SEED,verbose=0)
print(f"Best: {automl.best_estimator}")
yp_fl=automl.predict(X_vfl)
results["FLAML"]={"Accuracy":accuracy_score(y_val,yp_fl),"Macro-F1":f1_score(y_val,yp_fl,average="macro")}
print(results["FLAML"])

## 21. Top 3 Model Selection

In [ ]:
try:
    from lightgbm import LGBMClassifier; has_lgbm=True
except ImportError: has_lgbm=False
try:
    from xgboost import XGBClassifier; has_xgb=True
except ImportError: has_xgb=False
top3={}
if has_lgbm: top3["LightGBM"]=LGBMClassifier(n_estimators=300,learning_rate=0.05,max_depth=4,random_state=SEED,verbose=-1,n_jobs=-1)
else:
    from sklearn.ensemble import ExtraTreesClassifier
    top3["ExtraTrees"]=ExtraTreesClassifier(n_estimators=300,random_state=SEED,n_jobs=-1)
if has_xgb: top3["XGBoost"]=XGBClassifier(n_estimators=300,learning_rate=0.05,max_depth=4,random_state=SEED,verbosity=0,n_jobs=-1,eval_metric="mlogloss")
else:
    from sklearn.ensemble import AdaBoostClassifier
    top3["AdaBoost"]=AdaBoostClassifier(n_estimators=200,random_state=SEED)
top3["GradBoosting"]=GradientBoostingClassifier(n_estimators=200,learning_rate=0.05,max_depth=3,random_state=SEED)
print(f"Top 3: {list(top3.keys())}")

## 22. Final Training and Evaluation of Top 3

In [ ]:
X_tr_t=preprocessor.fit_transform(X_train); X_te_t=preprocessor.transform(X_test)
final={}
for name,mdl in top3.items():
    mdl.fit(X_tr_t,y_train); yp=mdl.predict(X_te_t)
    final[name]={"Accuracy":accuracy_score(y_test,yp),"Macro-F1":f1_score(y_test,yp,average="macro")}
    print(f"\n{name}:"); print(classification_report(y_test,yp,target_names=le.classes_))
pd.DataFrame(final).T

In [ ]:
fig,axes=plt.subplots(1,len(top3),figsize=(5*len(top3),4))
if len(top3)==1: axes=[axes]
for ax,(n,m) in zip(axes,top3.items()):
    ConfusionMatrixDisplay.from_predictions(y_test,m.predict(X_te_t),ax=ax,cmap="Blues",display_labels=le.classes_); ax.set_title(n)
plt.tight_layout(); plt.show()

## 23. Error Analysis

In [ ]:
bm=list(top3.values())[0]; ypb=bm.predict(X_te_t)
misclass = (y_test != ypb).sum()
print(f"Misclassified: {misclass}/{len(y_test)}")
if misclass > 0:
    idx = np.where(y_test != ypb)[0]
    td = X_test.iloc[idx].copy()
    td["true"] = le.inverse_transform(y_test[idx])
    td["pred"] = le.inverse_transform(ypb[idx])
    print(td.head(10))

## 24. Interpretation and Business Insight

Na_to_K ratio dominates. BP separates drugA/B from drugC/X. Cholesterol refines within BP groups. Near-deterministic rules - a decision tree captures them perfectly.

## 25. Limitations

1. Only 200 rows
2. Synthetic and deterministic
3. No interaction effects
4. No side effects considered
5. No temporal data

## 26. How to Improve This Project

1. Cross-validation (essential with 200 rows)
2. Decision tree (captures rules perfectly)
3. Extract decision rules
4. Add real clinical features
5. SHAP for per-patient explanations

## 27. Production Considerations

- Clinical decision support only
- FDA/regulatory validation required
- Explainability for physicians
- Liability for wrong recommendations
- Outcome monitoring

## 28. Common Mistakes

1. Not using ordinal encoding for BP and Cholesterol
2. Using complex models when decision tree suffices
3. Not stratifying (small class counts)
4. Overfitting on 200 rows
5. Ignoring Na_to_K dominance

## 29. Mini Challenge / Exercises

1. DecisionTree max_depth=3 accuracy?
2. Extract decision rules
3. Remove Na_to_K - performance drop?
4. 5-fold CV stability?
5. Best Na_to_K threshold for drugY?

## 30. Final Summary / Key Takeaways

| Aspect | Detail |
|---|---|
| Task | 5-class classification - drug type |
| Dataset | 200 patients, 5 features |
| Difficulty | Easy - deterministic |
| Best metric | Macro-F1 |
| Baseline | ~45% |
| Key insight | Na_to_K dominates |
| Limitation | Tiny, synthetic |

**What you learned:**
- Multiclass with ordinal features
- OrdinalEncoder vs OneHotEncoder
- Simple models can outperform complex ones
- Full benchmark -> AutoML -> top 3 pipeline